In [1]:
import pandas as pd
import numpy as np
import joblib

In [3]:
# Cargar modelos
har_ampliado_pkg = joblib.load("../models/HAR_ampliado_Ridge.joblib")
har_tuned_pkg = joblib.load("../models/HAR_ampliado_weighted_TUNED.joblib")

# Cargar dataset final
df_final = pd.read_csv("../data/processed/final_dataset_har.csv")

print("Dataset cargado:", df_final.shape)
print("\nColumnas:")
print(df_final.columns.tolist()[:20])

Dataset cargado: (6735, 15)

Columnas:
['date', 'y_log', 'log_spy_rv_5d', 'log_spy_rv_10d', 'log_spy_rv_20d', 'log_vix_close', 'vix_chg_1d', 'log_qqq_rv_5d', 'spy_ret_neg', 'log_spy_hl_range', 'log_abs_spy_ret', 'log_abs_spy_gap', 'log_vix_hl_range', 'log_vix_close_lag1', 'log_spy_hl_range_lag1']


In [4]:
features = har_tuned_pkg["features"]   # también podrías usar har_ampliado_pkg["features"]

# Comprobar si faltan columnas
missing_features = [col for col in features if col not in df_final.columns]
print("Features que faltan:", missing_features)

# Construir X
X = df_final[features].copy()

print("Shape de X:", X.shape)
X.head()

Features que faltan: []
Shape de X: (6735, 13)


,log_spy_rv_5d,log_spy_rv_10d,log_spy_rv_20d,log_vix_close,vix_chg_1d,log_qqq_rv_5d,spy_ret_neg,log_spy_hl_range,log_abs_spy_ret,log_abs_spy_gap,log_vix_hl_range,log_vix_close_lag1,log_spy_hl_range_lag1
0,-1.897602,-1.543374,-1.485415,3.117507,-0.009645,-1.259367,0.000000,-3.927201,-4.368274,-8.357217,-2.712042,3.127199,-4.198375
1,-1.799358,-1.612072,-1.495985,3.080533,-0.036299,-1.303378,0.000000,-4.259210,-8.369809,-5.804900,-3.041311,3.117507,-3.927201
2,-2.258230,-1.655893,-1.504929,3.087856,0.007350,-2.058706,0.000000,-3.755713,-4.541442,-4.563421,-2.009446,3.080533,-4.259210
3,-2.068946,-1.704608,-1.514612,3.123686,0.036480,-1.861607,-0.006419,-4.231111,-5.048480,-7.687517,-2.831016,3.087856,-3.755713
4,-1.638435,-1.585125,-1.474522,3.216473,0.097228,-1.299943,-0.016844,-3.675128,-4.083786,-5.378512,-2.579896,3.123686,-4.231111


In [6]:
pred_log_har = har_ampliado_pkg["model"].predict(X)
pred_log_har_tuned = har_tuned_pkg["model"].predict(X)

# Volver a escala original

pred_har = np.exp(pred_log_har)
pred_har_tuned = np.exp(pred_log_har_tuned)

print("Predicciones HAR (original):", pred_har[:5])
print("Predicciones HAR (tuned):", pred_har_tuned[:5])

Predicciones HAR (original): [0.16377252 0.17070897 0.16473866 0.16850749 0.19533856]
Predicciones HAR (tuned): [0.19567427 0.19588598 0.19609377 0.18748871 0.22503042]


/Users/quique/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/quique/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/quique/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/quique/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/quique/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/quique/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_base.py:279: Runti

In [8]:
pred_df = df_final.copy()

pred_df["pred_HAR_ampliado_Ridge"] = pred_har
pred_df["pred_HAR_ampliado_weighted_TUNED"] = pred_har_tuned

# Si existe la fecha, mejor verla primero
cols_show = []
if "date" in pred_df.columns:
    cols_show.append("date")

# Si existe target real
if "spy_future_rv_5d" in pred_df.columns:
    cols_show.append("spy_future_rv_5d")

cols_show += ["pred_HAR_ampliado_Ridge", "pred_HAR_ampliado_weighted_TUNED"]

print(pred_df[cols_show].head(10))

         date  pred_HAR_ampliado_Ridge  pred_HAR_ampliado_weighted_TUNED
0  1999-04-08                 0.163773                          0.195674
1  1999-04-09                 0.170709                          0.195886
2  1999-04-12                 0.164739                          0.196094
3  1999-04-13                 0.168507                          0.187489
4  1999-04-14                 0.195339                          0.225030
5  1999-04-15                 0.188133                          0.221006
6  1999-04-16                 0.170192                          0.189704
7  1999-04-19                 0.230656                          0.295857
8  1999-04-20                 0.195347                          0.234658
9  1999-04-21                 0.182072                          0.224337


In [9]:
def risk_group_3(x, t1=0.13, t2=0.20):
    if pd.isna(x):
        return np.nan
    elif x < t1:
        return "mantener"
    elif x < t2:
        return "reducir"
    else:
        return "reducir_fuerte"

pred_df["risk_HAR_ampliado_Ridge"] = pred_df["pred_HAR_ampliado_Ridge"].apply(risk_group_3)
pred_df["risk_HAR_ampliado_weighted_TUNED"] = pred_df["pred_HAR_ampliado_weighted_TUNED"].apply(risk_group_3)

cols_show_risk = []
if "date" in pred_df.columns:
    cols_show_risk.append("date")

cols_show_risk += [
    "pred_HAR_ampliado_Ridge",
    "risk_HAR_ampliado_Ridge",
    "pred_HAR_ampliado_weighted_TUNED",
    "risk_HAR_ampliado_weighted_TUNED"
]

print(pred_df[cols_show_risk].head(10))

         date  pred_HAR_ampliado_Ridge risk_HAR_ampliado_Ridge  \
0  1999-04-08                 0.163773                 reducir   
1  1999-04-09                 0.170709                 reducir   
2  1999-04-12                 0.164739                 reducir   
3  1999-04-13                 0.168507                 reducir   
4  1999-04-14                 0.195339                 reducir   
5  1999-04-15                 0.188133                 reducir   
6  1999-04-16                 0.170192                 reducir   
7  1999-04-19                 0.230656          reducir_fuerte   
8  1999-04-20                 0.195347                 reducir   
9  1999-04-21                 0.182072                 reducir   

   pred_HAR_ampliado_weighted_TUNED risk_HAR_ampliado_weighted_TUNED  
0                          0.195674                          reducir  
1                          0.195886                          reducir  
2                          0.196094                         

In [10]:
action_map = {
    "mantener": 1.00,
    "reducir": 0.60,
    "reducir_fuerte": 0.30
}

pred_df["exposure_HAR_ampliado_Ridge"] = pred_df["risk_HAR_ampliado_Ridge"].map(action_map)
pred_df["exposure_HAR_ampliado_weighted_TUNED"] = pred_df["risk_HAR_ampliado_weighted_TUNED"].map(action_map)

cols_show_expo = []
if "date" in pred_df.columns:
    cols_show_expo.append("date")

cols_show_expo += [
    "risk_HAR_ampliado_Ridge",
    "exposure_HAR_ampliado_Ridge",
    "risk_HAR_ampliado_weighted_TUNED",
    "exposure_HAR_ampliado_weighted_TUNED"
]

print(pred_df[cols_show_expo].head(10))

         date risk_HAR_ampliado_Ridge  exposure_HAR_ampliado_Ridge  \
0  1999-04-08                 reducir                          0.6   
1  1999-04-09                 reducir                          0.6   
2  1999-04-12                 reducir                          0.6   
3  1999-04-13                 reducir                          0.6   
4  1999-04-14                 reducir                          0.6   
5  1999-04-15                 reducir                          0.6   
6  1999-04-16                 reducir                          0.6   
7  1999-04-19          reducir_fuerte                          0.3   
8  1999-04-20                 reducir                          0.6   
9  1999-04-21                 reducir                          0.6   

  risk_HAR_ampliado_weighted_TUNED  exposure_HAR_ampliado_weighted_TUNED  
0                          reducir                                   0.6  
1                          reducir                                   0.6  
2   

In [ ]:
pred_df["risk_HAR_ampliado_Ridge"].value_counts()

risk_HAR_ampliado_Ridge
mantener          3748
reducir           1909
reducir_fuerte    1078
Name: count, dtype: int64

In [12]:
pred_df["risk_HAR_ampliado_weighted_TUNED"].value_counts()

risk_HAR_ampliado_weighted_TUNED
mantener          3133
reducir           1935
reducir_fuerte    1667
Name: count, dtype: int64